In [3]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display, HTML

# ==========================================
# Clean Publication-Ready Theme
# ==========================================
plt.rcParams['font.family'] = 'serif'
plt.rcParams['figure.facecolor'] = '#FFFFFF'  # White background
plt.rcParams['axes.facecolor'] = '#FFFFFF'
plt.rcParams['text.color'] = '#000000'        # Pure black text
plt.rcParams['axes.labelcolor'] = '#000000'
plt.rcParams['axes.edgecolor'] = '#000000'
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.color'] = '#F0F0F0'

# Palette
COLOR_SIM = '#0f4c81' 
COLOR_BASE = '#95a5a6'
COLOR_NEG = '#c0392b'

@interact(
    theta=FloatSlider(value=0.03, min=0.01, max=0.1, step=0.001, description='θ (R&D Prod)'),
    L=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='L (Labor)'),
    alpha=FloatSlider(value=0.3, min=0.1, max=0.9, step=0.01, description='α (Cap Share)'),
    mu=FloatSlider(value=1.6, min=1.01, max=3.0, step=0.01, description='μ (Markup)'),
    r=FloatSlider(value=0.04, min=0.01, max=0.15, step=0.01, description='r (Interest)'),
    T=IntSlider(value=100, min=20, max=200, step=10, description='Years (T)')
)
def romer_simulation(theta, L, alpha, mu, r, T):

    # === CORE ECONOMIC LOGIC ===
    g_raw = theta * L - ((1 - alpha) / alpha) * (mu / (mu - 1)) * r
    g_current = max(0.0001, g_raw)
    L_Y = ((1 - alpha) / (alpha * theta)) * (mu / (mu - 1)) * r
    L_Y = min(L * 0.99, max(0.01 * L, L_Y))
    L_R = L - L_Y
    g_baseline = 0.02

    # Dashboard Header
    html_content = f"""
    <div style="font-family: Georgia, serif; max-width: 900px; border-bottom: 2px solid #000; padding-bottom: 10px; margin-bottom: 20px;">
        <h2 style="color: #000; font-weight: bold; margin-bottom: 5px;">Endogenous Growth Simulation (Romer Model)</h2>
        <div style="display: flex; gap: 40px;">
            <div>
                <p style="margin: 0; font-size: 12px; text-transform: uppercase; font-weight: bold; color: #555;">Growth Rate (g*)</p>
                <p style="font-size: 24px; font-weight: bold; color: #0f4c81; margin: 0;">{g_current*100:.2f}%</p>
            </div>
            <div>              
                <p style="color: #555555; margin: 0; font-size: 14px; text-transform: uppercase;">Labor Allocation</p>
                <p style="font-size: 18px; margin: 5px 0;"><b>Production (L_Y):</b> {L_Y:.2f}</p>
                <p style="font-size: 18px; margin: 0;"><b>R&D (L_R):</b> {L_R:.2f}</p>
            </div>
        </div>
    </div>
    """
    display(HTML(html_content))

    # === VISUALIZATION LAYOUT ===
    fig, axs = plt.subplots(2, 2, figsize=(15, 11))
    
    # Increased padding (wspace/hspace) to provide plenty of space between graphs
    fig.subplots_adjust(hspace=0.4, wspace=0.4)

    t = np.arange(0, T+1)
    Y_baseline = (1 + g_baseline) ** t
    Y_current = (1 + g_current) ** t

    # Apply style to all axes (remove top/right "box" spines)
    for ax in axs.flat:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # 1. Output Trajectory
    axs[0,0].fill_between(t, Y_current, color=COLOR_SIM, alpha=0.1)
    axs[0,0].plot(t, Y_baseline, linewidth=2, color=COLOR_BASE, linestyle='--', label='Baseline Path')
    axs[0,0].plot(t, Y_current, linewidth=3, color=COLOR_SIM, label='Simulated Path')
    axs[0,0].set_title('Figure 1: Output Trajectory (Log Scale)', loc='left', pad=15, fontweight='bold')
    axs[0,0].set_xlabel('Time Horizon (Years)', fontweight='bold')
    axs[0,0].set_ylabel('Output (Y)', fontweight='bold')
    axs[0,0].set_yscale('log')
    axs[0,0].legend(frameon=False, loc='upper left')

    # 2. Labor Allocation (CONVERTED TO PIE CHART)
    labels = ['Production (L_Y)', 'R&D (L_R)']
    sizes = [L_Y, L_R]
    axs[0,1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, 
                colors=[COLOR_BASE, COLOR_SIM], 
                textprops={'fontweight': 'bold', 'color': 'black'},
                wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    axs[0,1].set_title('Figure 2: Equilibrium Labor Allocation', loc='left', pad=15, fontweight='bold')
    # Remove all spines/axes for the pie chart
    axs[0,1].axis('off')

    # 3. Growth Comparison
    bars = axs[1,0].bar(['Baseline', 'Simulation'], [g_baseline*100, g_current*100], 
                        color=[COLOR_BASE, COLOR_SIM], width=0.5, edgecolor='black')
    axs[1,0].set_title('Figure 3: Long-Run Growth Variance', loc='left', pad=15, fontweight='bold')
    axs[1,0].set_ylabel('Annual Growth (%)', fontweight='bold')
    axs[1,0].set_xlabel('Scenario', fontweight='bold')
    for bar in bars:
        yval = bar.get_height()
        axs[1,0].text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f}%', 
                      ha='center', va='bottom', fontweight='bold', color='black')

    # 4. Sensitivity Analysis
    param_names = ['θ (Productivity)', 'μ (Markup)', 'r (Interest)', 'α (Cap Share)', 'L (Labor)']
    changes = [0.01, 0.2, -0.01, 0.05, 0.5]
    g_sens = []
    for i, delta in enumerate(changes):
        if i == 0:   gs = (theta+delta)*L - ((1-alpha)/alpha)*(mu/(mu-1))*r
        elif i == 1: gs = theta*L - ((1-alpha)/alpha)*((mu+delta)/(mu+delta-1))*r
        elif i == 2: gs = theta*L - ((1-alpha)/alpha)*(mu/(mu-1))*(r+delta)
        elif i == 3: gs = theta*L - ((1-(alpha+delta))/(alpha+delta))*(mu/(mu-1))*r
        else:        gs = theta*(L+delta) - ((1-alpha)/alpha)*(mu/(mu-1))*r
        g_sens.append((gs - g_raw)*100)

    y_pos = np.arange(len(param_names))
    axs[1,1].hlines(y=y_pos, xmin=0, xmax=g_sens, color=COLOR_SIM, linewidth=3)
    # Color points based on positive or negative impact
    pt_colors = [COLOR_SIM if val > 0 else COLOR_NEG for val in g_sens]
    axs[1,1].scatter(g_sens, y_pos, color=pt_colors, s=80, edgecolor='black', zorder=3)
    axs[1,1].set_yticks(y_pos)
    axs[1,1].set_yticklabels(param_names, fontweight='bold')
    axs[1,1].set_title('Figure 4: Which parameter affects Growth Most?', loc='left', pad=15, fontweight='bold')
    axs[1,1].set_xlabel('Percentage Points', fontweight='bold')
    axs[1,1].axvline(0, color='black', linewidth=1)

    plt.show()

interactive(children=(FloatSlider(value=0.03, description='θ (R&D Prod)', max=0.1, min=0.01, step=0.001), Floa…